# Load Multiple-Target Attention Task Data

## Section 1 – Load All Data (Single & Multiple / Lab & Game)

In [ ]:
import os, glob, re, ast
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import pingouin as pg
from IPython.display import display

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.dpi": 110, "axes.titlesize": 12})

BASE_DIR   = os.path.dirname(os.path.abspath("__file__"))
DATA_DIR   = os.path.join(BASE_DIR, "data_brsm")

SINGLE_LAB   = os.path.join(DATA_DIR, "single",   "lab")
SINGLE_PHONE = os.path.join(DATA_DIR, "single",   "phone")
MULTI_LAB    = os.path.join(DATA_DIR, "multiple",  "lab")
MULTI_PHONE  = os.path.join(DATA_DIR, "multiple",  "phone")

def _pid(path):
    m = re.match(r"^(\d+)", os.path.basename(path))
    return m.group(1) if m else os.path.basename(path)

def load_folder(folder):
    files = sorted(glob.glob(os.path.join(folder, "*.csv")))
    return {_pid(f): pd.read_csv(f) for f in files}

single_lab_raw   = load_folder(SINGLE_LAB)
single_phone_raw = load_folder(SINGLE_PHONE)
multi_lab_raw    = load_folder(MULTI_LAB)
multi_phone_raw  = load_folder(MULTI_PHONE)

print(f"Single - Lab: {sorted(single_lab_raw.keys(), key=int)}")
print(f"Single - Game: {sorted(single_phone_raw.keys(), key=int)}")
print(f"Multi - Lab: {sorted(multi_lab_raw.keys(), key=int)}")
print(f"Multi - Game: {sorted(multi_phone_raw.keys(), key=int)}")

: 

## Section 2 – Feature Extraction & Per-Participant Summary

In [ ]:
import ast as _ast

def _parse(v):
    try: return _ast.literal_eval(str(v).strip())
    except: return []

print("=" * 70)
print("  DIAGNOSTIC - Lab accuracy columns")
print("=" * 70)

print("\n--- SINGLE TARGET LAB ---")
print("Relevant columns in a sample file:")
sample_single = next(iter(single_lab_raw.values()))
trial_rows_s  = sample_single.dropna(subset=["trial.started"])
print("  Columns present:", [c for c in ["target_col", "mouse.clicked_name", "feedback.stopped", "fb.stopped", "notes"] if c in sample_single.columns])

clicked_unique_s = set()
for _, row in trial_rows_s.iterrows():
    for name in _parse(row.get("mouse.clicked_name", "[]")):
        clicked_unique_s.add(name)
print(f"  All unique clicked object names across all single-lab trials: {sorted(clicked_unique_s)}")
print(f"  Number of trial rows: {len(trial_rows_s)}")
print(f"  'fb.stopped' column exists: {'fb.stopped' in sample_single.columns}")

if "feedback.stopped" in sample_single.columns:
    fb_populated = trial_rows_s["feedback.stopped"].notna().sum()
    print(f"  'feedback.stopped' populated in {fb_populated}/{len(trial_rows_s)} trial rows")

print("\n--- MULTIPLE TARGET LAB ---")
sample_multi = next(iter(multi_lab_raw.values()))
trial_rows_m = sample_multi.dropna(subset=["trial.started"])
print("  Columns present:", [c for c in ["target_col", "mouse.clicked_name", "feedback.stopped", "fb.stopped", "notes"] if c in sample_multi.columns])

clicked_unique_m = set()
for _, row in trial_rows_m.iterrows():
    for name in _parse(row.get("mouse.clicked_name", "[]")):
        clicked_unique_m.add(name)
print(f"  All unique clicked object names: {sorted(clicked_unique_m)}")

click_counts = trial_rows_m["mouse.clicked_name"].apply(lambda v: len(_parse(v)))
print(f"  Clicks per trial — min: {click_counts.min()}, max: {click_counts.max()}, "
      f"mean: {click_counts.mean():.1f}")

if "fb.stopped" in sample_multi.columns:
    fb_m = trial_rows_m["fb.stopped"].notna().sum()
    print(f"  'fb.stopped' populated in {fb_m}/{len(trial_rows_m)} trial rows")
    cross = trial_rows_m.groupby("target_col")["fb.stopped"].apply(lambda x: x.notna().sum())
    print(f"  'fb.stopped' by target_col:\n{cross.to_string()}")

print("\n  POOLED across ALL multi-lab participants:")
total_trials, fb_trials, non_target_clicks = 0, 0, 0
for pid, df in multi_lab_raw.items():
    tr = df.dropna(subset=["trial.started"])
    total_trials += len(tr)
    if "fb.stopped" in df.columns:
        fb_trials += tr["fb.stopped"].notna().sum()
    for _, row in tr.iterrows():
        names = _parse(row.get("mouse.clicked_name", "[]"))
        non_target_clicks += sum(1 for n in names if "target" not in str(n).lower())
print(f"  Total trial rows: {total_trials}")
print(f"  Trials with fb.stopped populated: {fb_trials}  ({100*fb_trials/total_trials:.1f}%)")
print(f"  Clicks on non-target named objects: {non_target_clicks}")
print(f"\n  Conclusion: 'fb.stopped' is populated on {100*fb_trials/total_trials:.1f}% of trials.")

In [ ]:
def parse_list_col(val):
    if pd.isna(val):
        return []
    try:
        return ast.literal_eval(str(val).strip())
    except Exception:
        return []


def lab_features(raw_dict, condition="single"):
    records = []
    for pid, df in raw_dict.items():
        trials = df.dropna(subset=["trial.started"])
        rts, accs = [], []
        for _, row in trials.iterrows():
            times = parse_list_col(row.get("mouse.time", np.nan))
            if times:
                rts.append(float(times[0]) * 1000)
                if condition == "single":
                    names = parse_list_col(row.get("mouse.clicked_name", np.nan))
                    accs.append(1.0 if "target" in names else 0.0)
                else:
                    fb_val = row.get("fb.stopped", np.nan)
                    accs.append(0.0 if pd.notna(fb_val) else 1.0)
        if rts:
            records.append(dict(pid=int(pid), mean_RT=np.mean(rts), sd_RT=np.std(rts, ddof=1), mean_Acc=np.mean(accs) * 100, n_trials=len(rts)))
    return pd.DataFrame(records).sort_values("pid").reset_index(drop=True)


def game_features(raw_dict):
    records = []
    for pid, df in raw_dict.items():
        valid = df.dropna(subset=["InitialResponseTime(ms)", "SuccessRate(%)"])
        if len(valid):
            records.append(dict(pid=int(pid), mean_RT=valid["InitialResponseTime(ms)"].mean(), sd_RT=valid["InitialResponseTime(ms)"].std(ddof=1),
                                mean_Acc=valid["SuccessRate(%)"].mean(), n_trials=len(valid)))
    return pd.DataFrame(records).sort_values("pid").reset_index(drop=True)


sg_lab  = lab_features(single_lab_raw, condition="single");   sg_lab["group"]  = "Single"; sg_lab["modality"]  = "Lab"
sg_game = game_features(single_phone_raw);                    sg_game["group"] = "Single"; sg_game["modality"] = "Game"
ml_lab  = lab_features(multi_lab_raw,  condition="multiple"); ml_lab["group"]  = "Multi";  ml_lab["modality"]  = "Lab"
ml_game = game_features(multi_phone_raw);                     ml_game["group"] = "Multi";  ml_game["modality"] = "Game"

print("Single lab accuracy source : mouse.clicked_name")
print("Multi  lab accuracy source : fb.stopped")
print("Game   accuracy source     : SuccessRate(%)")
print()
print("Single lab mean_Acc range :", sg_lab.mean_Acc.min(), "-", sg_lab.mean_Acc.max())
print("Multi  lab mean_Acc range :", ml_lab.mean_Acc.min(), "-", ml_lab.mean_Acc.max())
print()
print(ml_lab[["pid","mean_Acc","n_trials"]].to_string(index=False))

long_df = pd.concat([sg_lab, sg_game, ml_lab, ml_game], ignore_index=True)
long_df["group"]    = pd.Categorical(long_df["group"],    categories=["Single","Multi"], ordered=True)
long_df["modality"] = pd.Categorical(long_df["modality"], categories=["Lab","Game"],    ordered=True)

wide_single = sg_lab[["pid","mean_RT","mean_Acc"]].rename(columns={"mean_RT":"RT_Lab","mean_Acc":"Acc_Lab"}) \
              .merge(sg_game[["pid","mean_RT","mean_Acc"]].rename(columns={"mean_RT":"RT_Game","mean_Acc":"Acc_Game"}), on="pid")
wide_single["group"] = "Single"

wide_multi  = ml_lab[["pid","mean_RT","mean_Acc"]].rename(columns={"mean_RT":"RT_Lab","mean_Acc":"Acc_Lab"}) \
              .merge(ml_game[["pid","mean_RT","mean_Acc"]].rename(columns={"mean_RT":"RT_Game","mean_Acc":"Acc_Game"}), on="pid")
wide_multi["group"]  = "Multi"

wide_df = pd.concat([wide_single, wide_multi], ignore_index=True)

print(f"\nLong table: {long_df.shape}   Wide table: {wide_df.shape}")
display(long_df.head(8))


## Section 3 – Descriptive Statistics

Mean (M) and Standard Deviation (SD) for Reaction Time and Accuracy across all four cells of the design.

In [ ]:
desc = (long_df.groupby(["group","modality"], observed=True)[["mean_RT","mean_Acc"]].agg(["mean","std","count"]).round(2))

desc.columns = ["RT_M","RT_SD","RT_n","Acc_M","Acc_SD","Acc_n"]

print("=" * 60)
print("  DESCRIPTIVE STATISTICS – Reaction Time (ms) & Accuracy (%)")
print("=" * 60)
display(desc)

pivot_rt  = long_df.pivot_table(values="mean_RT",  index="group", columns="modality", aggfunc=["mean","std"], observed=True).round(2)
pivot_acc = long_df.pivot_table(values="mean_Acc", index="group", columns="modality", aggfunc=["mean","std"], observed=True).round(2)

display(pivot_rt)
display(pivot_acc)

## Section 4 – RQ1: Concurrent Validity (Correlation)

In [ ]:
import warnings

def run_correlation(df_wide, label, dv="RT"):
    col_lab  = f"{dv}_Lab"
    col_game = f"{dv}_Game"
    x, y = df_wide[col_lab].values, df_wide[col_game].values

    if np.std(x) == 0 or np.std(y) == 0:
        print(f"\n  {label}  -  DV: {dv} SKIPPED (constant variable; no variance to correlate)")
        return np.nan, np.nan, np.nan, np.nan

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        r_p, p_p = stats.pearsonr(x, y)
        r_s, p_s = stats.spearmanr(x, y)

    n = len(x)
    print(f"\n{'─'*55}")
    print(f"  {label}  (n = {n})  –  DV: {dv}")
    print(f"{'─'*55}")
    print(f"  Pearson  r = {r_p:+.3f},  p = {p_p:.4f}  {'* p<.05' if p_p<.05 else 'ns'}")
    print(f"  Spearman ρ = {r_s:+.3f},  p = {p_s:.4f}  {'* p<.05' if p_s<.05 else 'ns'}")
    return r_p, p_p, r_s, p_s

print("=" * 55)
print("  CONCURRENT VALIDITY – Game vs Lab correlations")
print("=" * 55)

print("\n--- Reaction Time ---")
run_correlation(wide_df[wide_df.group=="Single"], "Single-Target Group", dv="RT")
run_correlation(wide_df[wide_df.group=="Multi"],  "Multi-Target Group",  dv="RT")

print("\n--- Accuracy ---")
print("  Note: Lab accuracy = 100% for all participants (zero variance).")
print("  Correlation with Lab Acc is undefined; Game accuracy shown below:")
for grp in ["Single","Multi"]:
    sub = wide_df[wide_df.group==grp]
    print(f"    {grp} Game Acc: M = {sub.Acc_Game.mean():.2f}%, SD = {sub.Acc_Game.std():.2f}%")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=False)
for ax, grp, color in zip(axes, ["Single","Multi"], ["steelblue","coral"]):
    sub = wide_df[wide_df.group==grp]
    ax.scatter(sub.RT_Lab, sub.RT_Game, color=color, edgecolors="k", s=70, zorder=3)
    m, b = np.polyfit(sub.RT_Lab, sub.RT_Game, 1)
    xl = np.array([sub.RT_Lab.min(), sub.RT_Lab.max()])
    ax.plot(xl, m*xl+b, color=color, lw=1.8, ls="--")
    r, p = stats.pearsonr(sub.RT_Lab, sub.RT_Game)
    ax.set_title(f"{grp}-Target  (r = {r:.3f}, p = {p:.3f})", fontsize=11)
    ax.set_xlabel("Lab Task  RT (ms)")
    ax.set_ylabel("Game  RT (ms)")
    ax.grid(alpha=.35)

fig.suptitle("RQ1 - Concurrent Validity: Game vs Lab RT", fontweight="bold")
plt.tight_layout()
plt.savefig("fig_rq1_correlation.png", bbox_inches="tight")
plt.show()

## Section 5 – RQ2 & RQ3: Mixed 2 × 2 ANOVA

- **RQ2:** Single vs Multiple Target Load (between factor)  
- **RQ3:** Game vs Lab Modality (within factor)  
- **Interaction:** Does the modality effect depend on target load?

In [ ]:
def run_mixed_anova(long, dv_col, dv_label):
    aov = pg.mixed_anova(
        data=long,
        dv=dv_col,
        within="modality",
        between="group",
        subject="pid"
    )
    print(f"\n{'='*65}")
    print(f"  Mixed ANOVA  –  DV: {dv_label}")
    print(f"{'='*65}")
    display(aov.round(4))

    p_col = next((c for c in aov.columns if "p-unc" in c or "pvalue" in c.lower() or c=="p"), None)
    if p_col:
        for _, row in aov.iterrows():
            p = row[p_col]
            print(f"  {row['Source']:<15} F({int(row['DF1'])},{int(row['DF2'])}) = {row['F']:.3f}, "
                  f"p = {p:.4f}  {'* sig' if p<.05 else 'ns'}")

    print(f"\n  Normality checks (Shapiro-Wilk per cell):")
    for grp in ["Single","Multi"]:
        for mod in ["Lab","Game"]:
            sub = long[(long.group==grp) & (long.modality==mod)][dv_col].dropna()
            if len(sub) >= 3 and sub.std() > 0:
                w, p = stats.shapiro(sub)
                print(f"    {grp} / {mod}: W = {w:.3f}, p = {p:.4f}  "
                      f"{'ok' if p>.05 else 'non-normal - consider non-parametric'}")
    return aov

long_df_reset = long_df.copy()
aov_rt  = run_mixed_anova(long_df_reset, "mean_RT",  "Reaction Time (ms)")
aov_acc = run_mixed_anova(long_df_reset, "mean_Acc", "Accuracy (%)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for ax, col, title in zip(axes, ["mean_RT","mean_Acc"], ["Mean RT (ms)","Mean Accuracy (%)"]):
    cell_means = (long_df_reset.groupby(["group","modality"], observed=True)[col].mean().reset_index())
    for grp, ls in [("Single","o-"),("Multi","s--")]:
        sub = cell_means[cell_means.group==grp].sort_values("modality")
        ax.plot(sub["modality"].astype(str), sub[col], ls, label=grp, ms=8, lw=2)
    ax.set_title(title)
    ax.set_xlabel("Modality")
    ax.set_ylabel(title)
    ax.legend(title="Target Load")
    ax.grid(alpha=.35)

fig.suptitle("RQ2 & RQ3 – Modality x Target Load Interaction Plot", fontweight="bold")
plt.tight_layout()
plt.savefig("fig_rq2_rq3_interaction.png", bbox_inches="tight")
plt.show()

## Section 6 – Pairwise Tests

In [ ]:
def cohens_d_paired(a, b):
    diff = np.array(a) - np.array(b)
    return diff.mean() / diff.std(ddof=1)

def cohens_d_indep(a, b):
    na, nb = len(a), len(b)
    pooled_sd = np.sqrt(((na-1)*np.std(a,ddof=1)**2 + (nb-1)*np.std(b,ddof=1)**2) / (na+nb-2))
    return (np.mean(a) - np.mean(b)) / pooled_sd

def report_ttest(label, t, p, d, df, kind="paired"):
    sig = "✓ p < .05" if p < .05 else "ns"
    print(f"  {label}")
    print(f"    t({df:.0f}) = {t:+.3f},  p = {p:.4f}  [{sig}],  Cohen's d = {d:.3f}")
    print()

print("=" * 65)
print("  SECTION 6a - PAIRED t-tests: Game vs Lab (within groups)")
print("  DV: Reaction Time (ms)")
print("=" * 65)
for grp in ["Single","Multi"]:
    sub = wide_df[wide_df.group==grp]
    t, p = stats.ttest_rel(sub.RT_Lab, sub.RT_Game)
    d    = cohens_d_paired(sub.RT_Lab, sub.RT_Game)
    report_ttest(f"{grp}-Target  (n = {len(sub)})", t, p, d, len(sub)-1)

print("  DV: Accuracy (%)")
print("-" * 65)
for grp in ["Single","Multi"]:
    sub = wide_df[wide_df.group==grp]
    t, p = stats.ttest_rel(sub.Acc_Lab, sub.Acc_Game)
    d    = cohens_d_paired(sub.Acc_Lab, sub.Acc_Game)
    report_ttest(f"{grp}-Target  (n = {len(sub)})", t, p, d, len(sub)-1)

print("\n" + "=" * 65)
print("  SECTION 6b  –  INDEPENDENT t-tests: Single vs Multi")
print("  DV: Reaction Time (ms)")
print("=" * 65)
for mod in ["Lab","Game"]:
    a = wide_df[wide_df.group=="Single"][f"RT_{mod}"].values
    b = wide_df[wide_df.group=="Multi" ][f"RT_{mod}"].values
    t, p = stats.ttest_ind(a, b, equal_var=False)  # Welch's
    d    = cohens_d_indep(a, b)
    df_w = (np.var(a,ddof=1)/len(a) + np.var(b,ddof=1)/len(b))**2 / \
           ((np.var(a,ddof=1)/len(a))**2/(len(a)-1) + (np.var(b,ddof=1)/len(b))**2/(len(b)-1))
    report_ttest(f"{mod} Task  (n_single={len(a)}, n_multi={len(b)})", t, p, d, df_w, kind="indep")

print("  DV: Accuracy (%)")
print("-" * 65)
for mod in ["Lab","Game"]:
    a = wide_df[wide_df.group=="Single"][f"Acc_{mod}"].values
    b = wide_df[wide_df.group=="Multi" ][f"Acc_{mod}"].values
    t, p = stats.ttest_ind(a, b, equal_var=False)
    d    = cohens_d_indep(a, b)
    df_w = (np.var(a,ddof=1)/len(a) + np.var(b,ddof=1)/len(b))**2 / \
           ((np.var(a,ddof=1)/len(a))**2/(len(a)-1) + (np.var(b,ddof=1)/len(b))**2/(len(b)-1))
    report_ttest(f"{mod} Task  (n_single={len(a)}, n_multi={len(b)})", t, p, d, df_w, kind="indep")

## Section 7 – RQ4: Effect of Level in the Game

In [ ]:
def build_level_df(raw_dict, group_label):
    frames = []
    for pid, df in raw_dict.items():
        sub = df[["Level","InitialResponseTime(ms)","SuccessRate(%)","HitRate(%)","FalseAlarms"]].copy()
        sub.columns = ["level","RT_ms","SuccessRate","HitRate","FalseAlarms"]
        sub["pid"]   = int(pid)
        sub["group"] = group_label
        frames.append(sub)
    return pd.concat(frames, ignore_index=True)

level_df = pd.concat([
    build_level_df(single_phone_raw, "Single"),
    build_level_df(multi_phone_raw,  "Multi")
], ignore_index=True)

print(f"Level observations: {len(level_df)}")
print(f"Levels present: {sorted(level_df.level.unique())}")

print("\n" + "=" * 60)
print("  Linear trend: RT_ms ~ Level  (OLS per group)")
print("=" * 60)
for grp in ["Single","Multi"]:
    sub = level_df[level_df.group==grp].dropna(subset=["level","RT_ms"])
    slope, intercept, r, p, se = stats.linregress(sub.level, sub.RT_ms)
    print(f"\n  {grp}-Target  (N obs = {len(sub)})")
    print(f"    β(level) = {slope:+.2f} ms/level, intercept = {intercept:.1f}")
    print(f"    r = {r:.3f},  r² = {r**2:.3f},  p = {p:.4f}  {'✓' if p<.05 else 'ns'}")

top10_groups = [level_df[level_df.level==lv]["RT_ms"].dropna().values
                for lv in range(1, 11)]
f_stat, p_aov = stats.f_oneway(*top10_groups)
print(f"\n  One-way ANOVA (levels 1–10, pooled): F = {f_stat:.3f}, p = {p_aov:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for ax, col, ylabel in zip(axes, ["RT_ms","SuccessRate"], ["RT (ms)","Success Rate (%)"]):
    for grp, color in [("Single","steelblue"),("Multi","coral")]:
        sub = level_df[level_df.group==grp].groupby("level")[col].agg(["mean","sem"]).reset_index()
        ax.plot(sub.level, sub["mean"], "o-", color=color, label=grp, ms=5, lw=1.8)
        ax.fill_between(sub.level, sub["mean"] - sub["sem"], sub["mean"] + sub["sem"], color=color, alpha=.18)
    ax.set_xlabel("Game Level")
    ax.set_ylabel(ylabel)
    ax.set_title(f"{ylabel} vs Level")
    ax.legend(title="Group")
    ax.grid(alpha=.35)

fig.suptitle("RQ4 – Effect of Game Level on Performance", fontweight="bold")
plt.tight_layout()
plt.savefig("fig_rq4_level_effect.png", bbox_inches="tight")
plt.show()

## Section 8 – Reliability

**Internal consistency** of the game scores is assessed via Cronbach's α computed over level-wise RT values for each participant (items = levels).  
An α ≥ .70 indicates acceptable internal consistency; ≥ .80 is good.

In [ ]:
def cronbach_alpha(df_items):
    df_items = df_items.dropna()
    k = df_items.shape[1]
    if k < 2:
        return np.nan
    item_vars = df_items.var(axis=0, ddof=1).sum()
    total_var = df_items.sum(axis=1).var(ddof=1)
    if total_var == 0:
        return np.nan
    return (k / (k - 1)) * (1 - item_vars / total_var)

def split_half_reliability(df_items):
    df_items = df_items.dropna()
    k = df_items.shape[1]
    half1 = df_items.iloc[:, :k//2].sum(axis=1)
    half2 = df_items.iloc[:, k//2:].sum(axis=1)
    r, _ = stats.pearsonr(half1, half2)
    sb = (2 * r) / (1 + r)
    return r, sb

print("=" * 65)
print("  RELIABILITY – Internal Consistency of Game RT (level = item)")
print("=" * 65)

for grp, raw_phone in [("Single", single_phone_raw), ("Multi", multi_phone_raw)]:
    sub = level_df[level_df.group==grp]
    pivot = sub.pivot_table(index="pid", columns="level", values="RT_ms", aggfunc="mean")

    thresh = pivot.shape[0] * 0.5
    pivot = pivot.dropna(thresh=int(thresh), axis=1).dropna(axis=0)

    alpha   = cronbach_alpha(pivot)
    r_sh, sb = split_half_reliability(pivot)

    print(f"\n  {grp}-Target Group  (n={pivot.shape[0]} participants, {pivot.shape[1]} levels as items)")
    print(f"    Chronbach's α (RT) = {alpha:.3f}  {'(good)' if alpha>=.8 else '(acceptable)' if alpha>=.7 else '(poor)'}")
    print(f"    Split-half r = {r_sh:.3f},  Spearman-Brown corrected = {sb:.3f}")

print("\n" + "─" * 65)
print("  Convergent Validity summary (Game vs Lab Pearson r)")
print("─" * 65)
for grp in ["Single","Multi"]:
    sub = wide_df[wide_df.group==grp]
    r, p = stats.pearsonr(sub.RT_Lab, sub.RT_Game)
    print(f"  {grp}: r_converge(RT) = {r:.3f}, p = {p:.4f}")

## Section 9 – Visualisations Summary

Four composite figures:
1. Bar + error chart — M ± SD across the 4 design cells
2. Box plots — RT and Accuracy distributions  
3. Heatmap — Pearson correlation matrix of all four cell scores  
4. Raincloud-style plot — individual data points with distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
palette = {"Lab":"#4878CF", "Game":"#D65F5F"}

for ax, col, title, unit in zip(axes, ["mean_RT","mean_Acc"], ["Reaction Time","Accuracy"], ["ms","%"]):
    cell_stats = (long_df.groupby(["group","modality"], observed=True)[col].agg(mean="mean", sd="std").reset_index())
    x = np.arange(2)
    width = 0.35
    for i, (mod, color) in enumerate([("Lab","#4878CF"),("Game","#D65F5F")]):
        sub_m = cell_stats[cell_stats.modality==mod].sort_values("group")
        bars = ax.bar(x + i*width, sub_m["mean"], width,
                      label=mod, color=color, edgecolor="k", linewidth=.6,
                      yerr=sub_m["sd"], capsize=5, error_kw=dict(lw=1.5))
    ax.set_xticks(x + width/2)
    ax.set_xticklabels(["Single","Multi"])
    ax.set_xlabel("Target Load")
    ax.set_ylabel(f"{title} ({unit})")
    ax.set_title(f"{title} across Design Cells")
    ax.legend(title="Modality")
    ax.grid(axis="y", alpha=.35)

fig.suptitle("Figure 1 – M ± SD across 4 Cells", fontweight="bold")
plt.tight_layout()
plt.savefig("fig1_bar_cells.png", bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
long_df["cell"] = long_df["group"].astype(str) + "\n" + long_df["modality"].astype(str)
order = ["Single\nLab","Single\nGame","Multi\nLab","Multi\nGame"]
pal   = ["#4878CF","#D65F5F","#4878CF","#D65F5F"]

for ax, col, title, unit in zip(axes, ["mean_RT","mean_Acc"], ["Reaction Time","Accuracy"], ["ms","%"]):
    sns.boxplot(data=long_df, x="cell", y=col, order=order, palette=pal, width=.5, flierprops=dict(ms=4), ax=ax)
    sns.stripplot(data=long_df, x="cell", y=col, order=order, color="black", size=4, alpha=.55, jitter=True, ax=ax)
    ax.set_xlabel("")
    ax.set_ylabel(f"{title} ({unit})")
    ax.set_title(f"{title} Distributions")
    ax.grid(axis="y", alpha=.35)

fig.suptitle("Figure 2 – RT & Accuracy Distributions by Design Cell", fontweight="bold")
plt.tight_layout()
plt.savefig("fig2_boxplots.png", bbox_inches="tight")
plt.show()

corr_cols = ["RT_Lab","RT_Game","Acc_Lab","Acc_Game"]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, grp in zip(axes, ["Single","Multi"]):
    sub = wide_df[wide_df.group==grp][corr_cols]
    corr_mat = sub.corr()
    sns.heatmap(corr_mat, annot=True, fmt=".2f", cmap="coolwarm",
                center=0, vmin=-1, vmax=1, ax=ax,
                xticklabels=corr_cols, yticklabels=corr_cols,
                linewidths=.5, square=True)
    ax.set_title(f"{grp}-Target Group")

fig.suptitle("Figure 3 – Pearson Correlation Heatmap", fontweight="bold")
plt.tight_layout()
plt.savefig("fig3_corr_heatmap.png", bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, col, title, unit in zip(axes,
                                  ["mean_RT","mean_Acc"],
                                  ["Reaction Time","Accuracy"],
                                  ["ms","%"]):
    sns.violinplot(data=long_df, x="cell", y=col, order=order,
                   palette=pal, inner=None, cut=0, alpha=.6, ax=ax)
    sns.stripplot(data=long_df, x="cell", y=col, order=order,
                  color="black", size=5, alpha=.7, jitter=True, ax=ax)
    cell_means = long_df.groupby("cell")[col].mean()
    for xi, cell in enumerate(order):
        if cell in cell_means.index:
            ax.hlines(cell_means[cell], xi-.2, xi+.2, colors="white", lw=2.5, zorder=5)
    ax.set_xlabel("")
    ax.set_ylabel(f"{title} ({unit})")
    ax.set_title(f"{title} — Raincloud")
    ax.grid(axis="y", alpha=.35)

fig.suptitle("Figure 4 – Raincloud Plot (violin + individual scores)", fontweight="bold")
plt.tight_layout()
plt.savefig("fig4_raincloud.png", bbox_inches="tight")
plt.show()